# 03 - PhoBERT
- văn bản cột `text_phobert`
- chạy `python -m src.models.train_transformer --model phobert`

In [ ]:
import re, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

from src.config import (LABELS, DATA_DIR, MODELS_DIR, RESULTS_METRICS, RESULTS_FIGURES, RESULTS_PREDICTIONS, PREPROCESS_PRESETS, text_col)
print("ROOT =", ROOT)

In [ ]:
MODEL = "phobert"
from src.config import TRANSFORMER_COMMON, TRANSFORMERS
print("Preset tiền xử lý:", PREPROCESS_PRESETS[MODEL])
display(pd.Series({**TRANSFORMER_COMMON, **TRANSFORMERS[MODEL]}, name="cấu hình"))

d = pd.read_csv(DATA_DIR / "train.csv").sample(5, random_state=7)
display(d[["free_text", text_col(MODEL)]])

## 1. Nạp kết quả

In [ ]:
metrics_path = RESULTS_METRICS / f"{MODEL}.json"
assert metrics_path.exists(), f"Chưa có kết quả. Hãy chạy phobert"
print("Đã có kết quả:", metrics_path.name)

## 2. Kết quả

In [ ]:
from src.evaluate import show_results
show_results(MODEL)

In [ ]:
r = json.load(open(metrics_path, encoding="utf-8"))
ex = r["extra"]
print(f"Mô hình: {ex['model']} | tham số: {ex['n_params']:,} | epoch tốt nhất: {ex['best_epoch']} | thời gian: {ex['seconds']}s")
display(pd.Series(ex["hyperparams"], name="siêu tham số"))

overall = pd.DataFrame({s: {k: r[s][k] for k in ("accuracy", "macro_f1", "weighted_f1")} for s in ("val", "test")}).round(4)
display(overall)
for s in ("val", "test"):
    print(f"    Theo từng lớp ({s})")
    display(pd.DataFrame(r[s]["per_class"]).T.round(4))
display(Image(filename=str(RESULTS_FIGURES / f"cm_{MODEL}_val.png")))

## 3. loss và macro-f1 val theo epoch

In [ ]:
hist = pd.DataFrame(ex["history"])
display(hist)

fig, ax1 = plt.subplots(figsize=(6.5, 3.6))
ax1.plot(hist.epoch, hist.train_loss, "o-", color="tab:blue")
ax1.set_xlabel("epoch")
ax1.set_ylabel("train loss", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(hist.epoch, hist.val_macro_f1, "s--", color="tab:orange")
ax2.set_ylabel("val macro-F1", color="tab:orange")
ax1.axvline(ex["best_epoch"], color="gray", ls=":")
ax1.set_title(f"{MODEL} – epoch tốt nhất = {ex['best_epoch']}")
plt.tight_layout()
plt.show()

## 4. So với baseline

In [ ]:
bpath = RESULTS_METRICS / "baseline.json"
if bpath.exists():
    rb = json.load(open(bpath, encoding="utf-8"))
    runs = {"baseline": rb, MODEL: r}
    cmp = pd.DataFrame({
        name: {"val macro-F1": x["val"]["macro_f1"], "test macro-F1": x["test"]["macro_f1"], "test acc": x["test"]["accuracy"],
               **{f"test F1 {c}": m["f1"] for c, m in x["test"]["per_class"].items()}}
        for name, x in runs.items()
    }).round(4)
    cmp["chênh lệch (so với baseline)"] = (cmp[MODEL] - cmp["baseline"]).round(4)
    display(cmp)
else:
    print("Chưa có kết quả baseline để so sánh")

## 5. Độ tin cậy (softmax)

In [ ]:
pred = pd.read_csv(RESULTS_PREDICTIONS / f"{MODEL}_test.csv")
print("Độ tin cậy trung bình khi đoán đúng:", round(pred[pred.correct].confidence.mean(), 3))
print("Độ tin cậy trung bình khi đoán sai :", round(pred[~pred.correct].confidence.mean(), 3))
print("Số câu sai có độ tin cậy > 0.8      :", int(((~pred.correct) & (pred.confidence > 0.8)).sum()), "/", int((~pred.correct).sum()))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.hist([pred[pred.correct].confidence, pred[~pred.correct].confidence], bins=20, label=["đúng", "sai"], stacked=True)
ax.set_xlabel("độ tin cậy")
ax.set_ylabel("số câu")
ax.legend()
ax.set_title(f"{MODEL} – phân bố độ tin cậy trên test")
plt.tight_layout()
plt.show()